# Hybrid MPI+CUDA Programming

Most GPU clusters run **one MPI process per GPU**: MPI connects the processes, CUDA computes within each of them.
In the previous notebook, the second level of parallelism shared the memory of its process.
Here, it brings its own: every GPU has its memory, and the data a process communicates usually lives there.
The central question of MPI+CUDA programming is therefore: **how does a buffer in GPU memory get to the GPU of another process** — and what does each way require from the hardware, the system software and the MPI library?

This notebook builds on [GPU Programming with CUDA](cuda-primer.ipynb) and [Hybrid MPI+OpenMP Programming](hybrid-mpi-openmp.ipynb).

**After working through this notebook you will be able to**

* describe the components of a GPU node and the links between them, and read a node's topology with `nvidia-smi topo -m`,
* assign one GPU to every MPI process and place the processes close to their GPUs,
* name the datapaths for GPU data — staged through host memory, pipelined, GPUDirect RDMA, GPUDirect peer-to-peer, unified memory — and what each of them requires,
* find out whether your MPI library is CUDA-aware, and measure what the datapaths deliver on your system,
* synchronize CUDA streams and MPI calls correctly,
* organize halo exchanges and reductions of GPU data, and know the alternatives beyond MPI: NCCL, NVSHMEM, stream-aware MPI.

As before, prompts marked with 🤔 **Predict** ask you to think about the result *before* running the next cell.

## Working with this Notebook

The cells are compiled and run with the `%%mpicuda` cell magic.
It compiles with `nvcc`, which hands the host code to the MPI C++ compiler wrapper (`nvcc -ccbin mpicxx`), and starts the program with `mpirun`, like `%%mpi`:

| Option | Meaning |
|---|---|
| `-n N` | start `N` MPI processes (default: 4) |
| `-r` | the cell is a complete program — needed for every cell with a kernel |
| `-i mpicuda.h -f Icode` | include the helper header written below |
| `-e VAR=value` | set an environment variable for all processes, e.g. `PAMPI_CUDA_AWARE=0` |
| `-O VAR` | store the program output in the Python variable `VAR` |
| `-H wrapper` | the MPI C++ wrapper `nvcc` uses as host compiler (default: `mpicxx`, or the variable `ICE_MPI_CXX`) |
| `-L "…"` | replace the flags passed to `mpirun`, e.g. for placement |

**Requirements:** an NVIDIA GPU, the CUDA toolkit and MPI — on the cluster, start JupyterLab on a GPU node and load the CUDA and MPI modules.
If the MPI C++ wrapper has a different name, e.g. `mpiicpx` for Intel MPI, set `ICE_MPI_CXX` before starting JupyterLab.
Without `nvcc`, the cells are skipped with a message; read the code and the discussion anyway.

* **One GPU** is enough: all processes then share it.
* With **several GPUs in the node**, every process gets its own.
* With **two nodes**, you can measure the network paths, too — configure the launcher as described in the README, e.g. `ICE_MPI_LAUNCHER=srun`.

All cells work with every MPI library.
A **CUDA-aware** library is only needed for the paths that pass device pointers to MPI; the cells detect it and skip these paths otherwise.

Load the extension once per kernel session:

In [ ]:
%load_ext ice.magic

The helper header builds on `code/cuda-check.h` from the [CUDA primer](cuda-primer.ipynb) — `CUDA_CHECK()`, `CUDA_CHECK_LAUNCH()` and `GpuTimer` — and adds three functions for MPI+CUDA programs, explained in the following sections:

* `select_gpu()` assigns one GPU to every process,
* `mpi_cuda_aware()` asks the MPI library whether it accepts device pointers,
* `mpi_memory_kinds()` queries the memory kinds the library supports (MPI 4.1).

Run the cell to create the header:

In [ ]:
%%writefile code/mpicuda.h
// Helpers for the MPI+CUDA examples of the PAMPI notebooks.
#pragma once
#include <stdio.h>
#include <stdlib.h>
#include <mpi.h>
#if defined(OPEN_MPI) && OPEN_MPI
#include <mpi-ext.h>                // Open MPI extensions, among them MPIX_Query_cuda_support()
#endif
#include "cuda-check.h"             // CUDA_CHECK() and GpuTimer from the CUDA primer

// The rank of this process among the processes on its node: its rank in a shared-memory communicator.
static inline int node_rank(void) {
    MPI_Comm node;
    int local;
    MPI_Comm_split_type(MPI_COMM_WORLD, MPI_COMM_TYPE_SHARED, 0, MPI_INFO_NULL, &node);
    MPI_Comm_rank(node, &local);
    MPI_Comm_free(&node);
    return local;
}

// One GPU per process: process k of a node uses GPU k modulo the number of GPUs it can see.
// Call it before any other CUDA call, so that the CUDA context is created on the right GPU.
// Returns the number of the GPU.
static inline int select_gpu(void) {
    int ngpus;
    CUDA_CHECK(cudaGetDeviceCount(&ngpus));         // does not create a context
    const int gpu = node_rank() % ngpus;
    CUDA_CHECK(cudaSetDevice(gpu));
    return gpu;
}

// Can the MPI library work on device pointers? 1: yes, 0: no, -1: the library can't tell.
// *how describes where the answer comes from. The environment variable PAMPI_CUDA_AWARE=1 or 0
// overrides it - set it to 1 only if you know that your library is CUDA-aware (e.g. Cray MPICH
// with MPICH_GPU_SUPPORT_ENABLED=1): a library that is not crashes on device pointers.
static inline int mpi_cuda_aware(const char **how) {
    const char *env = getenv("PAMPI_CUDA_AWARE");
    if (env && *env) {
        *how = "set by PAMPI_CUDA_AWARE";
        return atoi(env) != 0;
    }
#if defined(MPIX_CUDA_AWARE_SUPPORT)
    *how = "Open MPI: MPIX_Query_cuda_support()";
    return MPIX_Query_cuda_support();
#elif defined(MPICH_NUMVERSION) && MPICH_NUMVERSION >= 40000000 && !defined(I_MPI_VERSION)
    int supported = 0;
    MPIX_GPU_query_support(MPIX_GPU_SUPPORT_CUDA, &supported);
    *how = "MPICH: MPIX_GPU_query_support()";
    return supported;
#else
    *how = "no query available for this MPI library";
    return -1;
#endif
}

// The memory kinds the library reports for MPI_COMM_WORLD (MPI 4.1, info key mpi_memory_alloc_kinds),
// e.g. "system,mpi,cuda:device,cuda:host,cuda:managed"
static inline void mpi_memory_kinds(char *kinds, int len) {
#if MPI_VERSION > 4 || (MPI_VERSION == 4 && MPI_SUBVERSION >= 1)
    MPI_Info info;
    int flag = 0, buflen = len;
    MPI_Comm_get_info(MPI_COMM_WORLD, &info);
    MPI_Info_get_string(info, "mpi_memory_alloc_kinds", &buflen, kinds, &flag);
    if (!flag)
        snprintf(kinds, len, "not reported");
    MPI_Info_free(&info);
#else
    snprintf(kinds, len, "n/a - the library implements MPI %d.%d, memory kinds came with MPI 4.1", MPI_VERSION, MPI_SUBVERSION);
#endif
}

## Anatomy of a GPU Node

A GPU node is a small network of its own.
The figure shows a typical layout with two CPU sockets and four GPUs; nodes like NVIDIA's DGX B200 have eight GPUs and eight network cards, but the same structure.

![A GPU node with two CPU sockets, each with its own host memory and a PCIe switch that connects two GPUs and one network card; the four GPUs are connected by NVLink; the network cards of all nodes are connected by the network](img/mpicuda-node.svg)

* Every **CPU socket** has its own host memory and its own PCIe lanes; here, each socket connects through a **PCIe switch** to two GPUs and one network card (NIC).
* The **GPUs** are connected with each other by **NVLink** — through NVSwitch chips with more than four GPUs.
* The **NICs** connect the node to the network. Ideally, every GPU has its own NIC on the same PCIe switch.

The links differ by more than two orders of magnitude (rounded peak values for a node with B200 GPUs):

| Link | Bandwidth | Relative to the network |
|---|---|---|
| HBM, within a GPU | 8 TB/s | 160× |
| NVLink 5, GPU ↔ GPU | 900 GB/s per direction | 18× |
| host memory, DDR5 with 12 channels per socket | ≈ 600 GB/s for both directions | 12× |
| NVLink-C2C, Grace CPU ↔ GPU in GH200 and GB200 systems | 450 GB/s per direction | 9× |
| PCIe 5.0 x16, GPU ↔ host or NIC | 64 GB/s per direction, ≈ 50–55 GB/s reached | ≈ 1× |
| InfiniBand NDR (400 Gbit/s), per NIC | 50 GB/s per direction | 1× |

Two consequences:

* Within a node, GPUs exchange data an order of magnitude faster with each other than with the CPUs or the network.
  Data that leaves the node should go **directly from the GPU to a NIC** — every detour through host memory crosses PCIe twice more.
* **Distance matters:** a GPU should send through the NIC on its own PCIe switch.
  The path from GPU 2 to NIC 0 crosses the inter-socket link, which is slower and shared with all other traffic between the sockets.

`nvidia-smi topo -m` prints the topology of a node as a matrix.
Its entries name the connection between two devices, from best to worst:

| Entry | Connection |
|---|---|
| `NV#` | NVLink, a bonded set of `#` links |
| `PIX` | at most one PCIe switch |
| `PXB` | several PCIe switches, but not through the CPU |
| `PHB` | through a PCIe host bridge, i.e. the CPU |
| `NODE` | between the PCIe host bridges of one ccNUMA domain |
| `SYS` | across the inter-socket link |

The columns *CPU Affinity* and *NUMA Affinity* list the cores and the ccNUMA domain closest to each GPU.

In [ ]:
# the topology of this node (needs the NVIDIA driver)
import shutil
import subprocess

if shutil.which('nvidia-smi'):
    print(subprocess.run(['nvidia-smi', 'topo', '-m'], capture_output=True, text=True).stdout)
else:
    print('nvidia-smi not found - run this cell on a GPU node')

> 🤔 Which GPUs of your node are connected by NVLink? Which GPU–NIC pairs are `PIX` or `PXB`? Keep the answers for Exercise 1.

## One Process per GPU

Several processes can share a GPU — the GPU then switches between their contexts, unless the *Multi-Process Service* (MPS) lets their kernels run side by side.
But one process per GPU is the rule: every process gets the whole GPU memory, and no time is lost switching.

Every process must select **its** GPU, and it must do so **before it uses CUDA**: the first CUDA call creates a *context* on the current device — by default device 0, so all processes would end up there.
There are two ways:

1. **In the program:** number the processes of a node with a shared-memory communicator (see Communicators, Groups and Shared-Memory Windows), and select GPU *node rank* modulo *number of GPUs* — this is `select_gpu()` in the header:
   ```c
   MPI_Comm_split_type(MPI_COMM_WORLD, MPI_COMM_TYPE_SHARED, 0, MPI_INFO_NULL, &node);
   MPI_Comm_rank(node, &local);
   cudaGetDeviceCount(&ngpus);
   cudaSetDevice(local % ngpus);
   ```
2. **By the launcher:** the environment variable `CUDA_VISIBLE_DEVICES` restricts a process to some of the GPUs, which then appear as devices 0, 1, ….
   Slurm sets it per task with `srun --gpus-per-task=1 --gpu-bind=closest`.
   With other launchers, a small wrapper script does it, e.g. for Open MPI, started as `mpirun -n 8 ./gpu-wrapper.sh ./a.out`:
   ```bash
   #!/bin/sh
   # Open MPI; MPICH (Hydra): $MPI_LOCALRANKID, Slurm: $SLURM_LOCALID
   export CUDA_VISIBLE_DEVICES=$OMPI_COMM_WORLD_LOCAL_RANK
   exec "$@"
   ```

The first way is portable and keeps all GPUs visible to all processes; the second one also works for programs you can't change.
Two caveats:

* Some CUDA-aware MPI libraries set up CUDA already in `MPI_Init()`; older versions of Open MPI and MVAPICH2 required the GPU to be selected **before** `MPI_Init()`.
  Then only the launcher, or the node rank from the launcher's environment variables, can be used.
* Restricting every process to one GPU with `CUDA_VISIBLE_DEVICES` can prevent the MPI library from using CUDA IPC between the processes of a node (see below) — it then silently stages through host memory.
  If the bandwidth between the GPUs of a node is unexpectedly low, make all GPUs visible and select in the program.

Like threads, processes must be **pinned close to their GPU**: the host side of a GPU — kernel launches, staging buffers, the MPI library — runs best on the cores of the socket the GPU is attached to (*CPU Affinity* in `nvidia-smi topo -m`).
Slurm's `--gpu-bind=closest` with `--cpu-bind=cores` takes care of that; with Open MPI, `--map-by numa` or `--map-by package` distributes the processes over the sockets — check that the node ranks then match the numbering of the GPUs.
A GPU node has far more cores than GPUs: every process can use the other cores of its socket with OpenMP threads — MPI+OpenMP+CUDA.

The following cell shows where every process runs and which GPU it uses, and asks the MPI library whether it is CUDA-aware.
Run it with as many processes as your node has GPUs, and with more.

In [ ]:
%%mpicuda -r -n 2 -i mpicuda.h -f Icode -o code/mpicuda-where.cu
#ifdef __linux__
#include <sched.h>                                  // sched_getcpu()
#endif

int main(int argc, char *argv[]) {
    MPI_Init(&argc, &argv);
    int rank, size;
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);

    const int local = node_rank();
    const int gpu = select_gpu();                   // first CUDA call apart from counting the GPUs

    cudaDeviceProp prop;
    CUDA_CHECK(cudaGetDeviceProperties(&prop, gpu));
    char bus[32];                                   // the PCI address identifies the GPU in nvidia-smi
    CUDA_CHECK(cudaDeviceGetPCIBusId(bus, sizeof(bus), gpu));
    int ngpus;
    CUDA_CHECK(cudaGetDeviceCount(&ngpus));

    char host[MPI_MAX_PROCESSOR_NAME];
    int len;
    MPI_Get_processor_name(host, &len);
    const char *visible = getenv("CUDA_VISIBLE_DEVICES");
#ifdef __linux__
    const int cpu = sched_getcpu();
#else
    const int cpu = -1;
#endif

    // one line per process, collected on rank 0 and printed in order
    char line[256];
    snprintf(line, sizeof(line), "rank %d on %s (node rank %d, CPU %d): GPU %d of %d [%s], %s, CUDA_VISIBLE_DEVICES=%s",
             rank, host, local, cpu, gpu, ngpus, bus, prop.name, visible ? visible : "(unset)");
    char *lines = rank == 0 ? (char *)malloc(size * sizeof(line)) : NULL;
    MPI_Gather(line, sizeof(line), MPI_CHAR, lines, sizeof(line), MPI_CHAR, 0, MPI_COMM_WORLD);

    if (rank == 0) {
        for (int r = 0; r < size; ++r)
            printf("%s\n", lines + r * sizeof(line));
        free(lines);

        const char *how;
        const int aware = mpi_cuda_aware(&how);
        char kinds[MPI_MAX_INFO_VAL + 1];
        mpi_memory_kinds(kinds, sizeof(kinds));
        printf("\nCUDA-aware MPI : %s (%s)\n", aware == 1 ? "yes" : aware == 0 ? "no" : "unknown", how);
        printf("memory kinds   : %s\n", kinds);
    }

    MPI_Finalize();
    return 0;
}

* With one GPU, all processes share GPU 0; with several, every process gets its own. Compare the PCI addresses with the output of `nvidia-smi topo -m` above — and the CPU of every process with the *CPU Affinity* of its GPU.
* **CUDA-aware?** The MPI standard says nothing about GPU memory: whether a library accepts device pointers depends on the library, how it was built, and how it is configured at runtime.
  Open MPI and MPICH can be asked with extensions (`MPIX_Query_cuda_support()`, `MPIX_GPU_query_support()`).
  Others, like Cray MPICH, can't — if you know that yours is CUDA-aware, add `-e PAMPI_CUDA_AWARE=1` to the cells below.
  Be careful: a library that is not CUDA-aware crashes on device pointers, typically with a segmentation fault.
* **Memory kinds** (MPI 4.1) standardize the question: the info key `mpi_memory_alloc_kinds` of a communicator, window or file lists the kinds of memory the library supports for it, e.g. `system,mpi,cuda:device,cuda:managed`.
  A program can request kinds when it creates a session (`MPI_Session_init()`); a library may then report only those it was asked for.
  Support in the libraries is just emerging.

## Datapaths for GPU Data

Process A has computed a buffer in GPU memory, process B needs it in its own GPU memory.
There are four datapaths:

![Four datapaths of a message from GPU memory of one process to GPU memory of another: staged through host memory with three transfers one after the other; pipelined staging in chunks, where the copies and the transfers of different chunks overlap; GPUDirect RDMA, where the network cards read and write GPU memory directly; GPUDirect peer-to-peer within a node, a single copy over NVLink or PCIe](img/mpicuda-datapaths.svg)

Plus one that hides the question: **unified memory**, where pages move on demand.
Which path a message takes depends on the hardware, the system software, the MPI library — and the code.

### Staged through Host Memory

The program copies the data to the host, sends it with MPI, and the receiver copies it to its GPU:

```c
cudaMemcpy(h_buf, d_buf, bytes, cudaMemcpyDeviceToHost);    // waits for the kernels in the default stream
MPI_Send(h_buf, count, MPI_DOUBLE, dest, tag, comm);
…
MPI_Recv(h_buf, count, MPI_DOUBLE, source, tag, comm, MPI_STATUS_IGNORE);
cudaMemcpy(d_buf, h_buf, bytes, cudaMemcpyHostToDevice);
```

* **Requires:** nothing — it works with every MPI library.
* **Costs:** two extra transfers over PCIe, one after the other.
  For small messages, each `cudaMemcpy()` adds several microseconds — more than an MPI latency; for large messages, the times of the three transfers add up.
* Use **pinned** host buffers (`cudaMallocHost()`): a copy from pageable memory is staged through a pinned buffer of the driver — one more copy.
* Large messages can be **pipelined**: in chunks, so that the copies of one chunk overlap with the transfer of another (path 2 in the figure, Exercise 2).

### CUDA-Aware MPI

A CUDA-aware MPI library accepts device pointers:

```c
MPI_Send(d_buf, count, MPI_DOUBLE, dest, tag, comm);        // d_buf from cudaMalloc()
```

With CUDA's *unified virtual addressing*, host and device buffers have disjoint addresses, so the library can find out where a buffer lives (`cuPointerGetAttributes()`) — and choose the best path:

* **Within a node — GPUDirect P2P:** the library maps the buffer of the other process into its own address space (**CUDA IPC**, `cudaIpcGetMemHandle()` and `cudaIpcOpenMemHandle()`), and a single copy moves the data from GPU to GPU, over NVLink or PCIe.
* **Between nodes — GPUDirect RDMA:** the NIC reads the data directly from GPU memory, and the receiving NIC writes it directly into GPU memory. The CPUs only start the transfer.
* **Fallback — pipelined staging:** where GPUDirect RDMA is unavailable or slow, e.g. because the GPU and the NIC are far apart, the library stages through pinned buffers itself, in chunks.
* **Small messages:** a `cudaMemcpy()` takes microseconds. With **GDRCopy**, the CPU reads and writes GPU memory directly through a PCIe mapping — much faster for a few bytes; UCX uses it for small messages.

CUDA-awareness is a property of how the library was **built and configured** — plus kernel modules and hardware:

| Component | Requirement | Check |
|---|---|---|
| MPI library | built with CUDA support | the cell above; Open MPI: `ompi_info --parsable --all \| grep mpi_built_with_cuda_support` |
| communication layer (UCX, libfabric) | built with CUDA support | `ucx_info -d \| grep cuda` lists the UCX transports `cuda_copy`, `cuda_ipc`, `gdr_copy` |
| GPUDirect RDMA | kernel module `nvidia_peermem` (or DMA-BUF with recent kernels and drivers), a NIC that supports it | `lsmod \| grep nvidia_peermem` |
| GDRCopy (optional) | kernel module `gdrdrv` and its library | `lsmod \| grep gdrdrv` |
| topology | GPU and NIC on one PCIe switch (`PIX`, `PXB`) for full GPUDirect RDMA bandwidth; NVLink or a common PCIe root for P2P | `nvidia-smi topo -m` |

The common libraries:

| Library | CUDA-aware use |
|---|---|
| Open MPI 5 | built `--with-cuda`, with a CUDA-enabled UCX (`--mca pml ucx` on InfiniBand); NVIDIA's HPC-X is such a build |
| MPICH 4 | built `--with-cuda`; GPU support is then enabled at runtime by default (`MPIR_CVAR_ENABLE_GPU`) |
| Cray MPICH | `export MPICH_GPU_SUPPORT_ENABLED=1` **and** link the GPU transport layer library (via the `craype-accel-*` module) |
| MVAPICH2-GDR | `export MV2_USE_CUDA=1` |
| Intel MPI | `export I_MPI_OFFLOAD=1` — designed for Intel GPUs first, check the documentation for NVIDIA GPUs |

On clusters, the MPI modules for GPU nodes are usually built this way — check the documentation of your computing center.

### Unified Memory

Memory from `cudaMallocManaged()` is valid on the host and on the device; the driver migrates its pages to where they are accessed.

* **Requires:** nothing special — even a library that is not CUDA-aware can read and write such a buffer: it is valid host memory.
  For transfers over the network, it is safer to treat it like device memory and use a CUDA-aware library.
* **Costs:** when MPI touches the buffer on the CPU, its pages migrate to the host, fault by fault — and the next kernel migrates them back. Convenient for porting, but every message pays for two migrations.
  A CUDA-aware library recognizes managed memory and may move it with the GPU's copy engines instead — faster, but still slower than plain device memory.
* On **coherent systems** like Grace Hopper (GH200) and Grace Blackwell (GB200), CPU and GPU share one address space over NVLink-C2C: even memory from `malloc()` is accessible from the GPU (on x86 systems with recent drivers, too, through *heterogeneous memory management*, HMM).
  The data still lives in one place — in the CPU's LPDDR or the GPU's HBM —, and that place decides the speed.

### Measuring the Datapaths

The following ping-pong sends messages of 8 B to 32 MiB between two processes.
The data lives in GPU memory on both sides — except for the host baseline:

| Column | Path |
|---|---|
| `host` | baseline: host buffers, no GPU involved |
| `pageable` | staged through `malloc()` memory |
| `pinned` | staged through `cudaMallocHost()` memory |
| `managed` | a `cudaMallocManaged()` buffer passed to MPI; after receiving, a kernel touches the data on the GPU, as the next kernel of a real code would |
| `device` | the device pointer passed to MPI — only with a CUDA-aware library |

The columns show the time per message — half a round trip — in microseconds; the plot below adds the bandwidth.
Run it with both processes on one GPU, on two GPUs of one node, and on two nodes if you can.

> 🤔 **Predict:** How much latency does staging add to a small message? Which path is fastest for 32 MiB, and by how much?

In [ ]:
%%mpicuda -r -n 2 -i mpicuda.h -f Icode -O pingpong -o code/mpicuda-pingpong.cu
#include <string.h>

// the datapaths: the data lives in GPU memory on both sides, except for the host baseline
enum { HOST, PAGEABLE, PINNED, MANAGED, DEVICE, NPATHS };
static const char *names[NPATHS] = { "host", "pageable", "pinned", "managed", "device" };

struct Buffers {
    char *dev;                                      // cudaMalloc(): the data on the GPU
    char *host;                                     // malloc(): pageable host memory
    char *pinned;                                   // cudaMallocHost(): pinned host memory
    char *managed;                                  // cudaMallocManaged(): migrates between CPU and GPU
};

// writes one byte of every page on the GPU - like a kernel that uses the received data,
// it moves the pages of a managed buffer to the GPU
__global__ void touch(char *buf, size_t bytes) {
    const size_t stride = (size_t)gridDim.x * blockDim.x * 4096;
    for (size_t i = (blockIdx.x * (size_t)blockDim.x + threadIdx.x) * 4096; i < bytes; i += stride)
        buf[i] += 1;
}

// sends a message whose data was produced on the GPU
static void send_msg(int path, Buffers *b, size_t bytes, int peer) {
    switch (path) {
    case HOST:                                      // baseline: no GPU involved
        MPI_Send(b->host, (int)bytes, MPI_BYTE, peer, 0, MPI_COMM_WORLD);
        break;
    case PAGEABLE:                                  // staged: GPU -> host, then MPI
    case PINNED: {
        char *staging = path == PINNED ? b->pinned : b->host;
        CUDA_CHECK(cudaMemcpy(staging, b->dev, bytes, cudaMemcpyDeviceToHost));
        MPI_Send(staging, (int)bytes, MPI_BYTE, peer, 0, MPI_COMM_WORLD);
        break;
    }
    case MANAGED:                                   // read on the CPU (or by a copy engine, if CUDA-aware)
        MPI_Send(b->managed, (int)bytes, MPI_BYTE, peer, 0, MPI_COMM_WORLD);
        break;
    case DEVICE:                                    // CUDA-aware MPI: the device pointer itself
        MPI_Send(b->dev, (int)bytes, MPI_BYTE, peer, 0, MPI_COMM_WORLD);
        break;
    }
}

// receives a message whose data is needed on the GPU
static void recv_msg(int path, Buffers *b, size_t bytes, int peer) {
    switch (path) {
    case HOST:
        MPI_Recv(b->host, (int)bytes, MPI_BYTE, peer, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
        break;
    case PAGEABLE:                                  // staged: MPI, then host -> GPU
    case PINNED: {
        char *staging = path == PINNED ? b->pinned : b->host;
        MPI_Recv(staging, (int)bytes, MPI_BYTE, peer, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
        CUDA_CHECK(cudaMemcpy(b->dev, staging, bytes, cudaMemcpyHostToDevice));
        break;
    }
    case MANAGED:                                   // written by MPI, then the GPU uses the data
        MPI_Recv(b->managed, (int)bytes, MPI_BYTE, peer, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
        touch<<<64, 256>>>(b->managed, bytes);
        CUDA_CHECK_LAUNCH();
        CUDA_CHECK(cudaDeviceSynchronize());
        break;
    case DEVICE:
        MPI_Recv(b->dev, (int)bytes, MPI_BYTE, peer, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
        break;
    }
}

int main(int argc, char *argv[]) {
    MPI_Init(&argc, &argv);
    int rank, size;
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);
    if (size != 2) {
        if (rank == 0)
            fprintf(stderr, "the ping-pong needs exactly 2 processes\n");
        MPI_Abort(MPI_COMM_WORLD, 1);
    }
    const int gpu = select_gpu(), peer = 1 - rank;

    const char *how;
    const int aware = mpi_cuda_aware(&how);

    // where are the two processes and their GPUs?
    char host[MPI_MAX_PROCESSOR_NAME] = { 0 }, peer_host[MPI_MAX_PROCESSOR_NAME] = { 0 };
    int len, peer_gpu;
    MPI_Get_processor_name(host, &len);
    MPI_Sendrecv(host, MPI_MAX_PROCESSOR_NAME, MPI_CHAR, peer, 0, peer_host, MPI_MAX_PROCESSOR_NAME, MPI_CHAR, peer, 0,
                 MPI_COMM_WORLD, MPI_STATUS_IGNORE);
    MPI_Sendrecv(&gpu, 1, MPI_INT, peer, 0, &peer_gpu, 1, MPI_INT, peer, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
    if (rank == 0) {
        if (strcmp(host, peer_host) != 0)
            printf("rank 0: GPU %d on %s, rank 1: GPU %d on %s - different nodes\n", gpu, host, peer_gpu, peer_host);
        else if (gpu == peer_gpu)
            printf("both ranks on %s share GPU %d\n", host, gpu);
        else {
            int p2p = 0;                            // can the two GPUs access each other's memory?
            CUDA_CHECK(cudaDeviceCanAccessPeer(&p2p, gpu, peer_gpu));
            printf("rank 0: GPU %d, rank 1: GPU %d on %s, peer access: %s\n", gpu, peer_gpu, host, p2p ? "yes" : "no");
        }
        printf("CUDA-aware MPI: %s (%s)%s\n\n", aware == 1 ? "yes" : aware == 0 ? "no" : "unknown", how,
               aware == 1 ? "" : " - the device path is skipped");
        printf("%10s", "bytes");
        for (int p = 0; p < NPATHS; ++p)
            printf(" %10s", names[p]);
        printf("   [time per message in us]\n");
    }

    const size_t max_bytes = (size_t)32 << 20;     // 32 MiB
    Buffers b;
    CUDA_CHECK(cudaMalloc(&b.dev, max_bytes));
    CUDA_CHECK(cudaMallocHost(&b.pinned, max_bytes));
    CUDA_CHECK(cudaMallocManaged(&b.managed, max_bytes));
    b.host = (char *)malloc(max_bytes);
    CUDA_CHECK(cudaMemset(b.dev, rank, max_bytes));
    memset(b.host, rank, max_bytes);
    memset(b.pinned, rank, max_bytes);
    memset(b.managed, rank, max_bytes);

    for (size_t bytes = 8; bytes <= max_bytes; bytes *= 4) {
        const size_t n = ((size_t)1 << 28) / bytes;                    // fewer repetitions for long messages
        const int reps = n > 1000 ? 1000 : n < 10 ? 10 : (int)n;
        if (rank == 0)
            printf("%10zu", bytes);
        for (int p = 0; p < NPATHS; ++p) {
            if (p == DEVICE && aware != 1) {
                if (rank == 0)
                    printf(" %10s", "n/a");
                continue;
            }
            if (p == MANAGED) {                     // the data starts on the GPU, as for the other paths
                touch<<<64, 256>>>(b.managed, bytes);
                CUDA_CHECK_LAUNCH();
                CUDA_CHECK(cudaDeviceSynchronize());
            }
            double start = 0.0;
            // 2 round trips to warm up: connections, registration of the buffers, CUDA IPC handles
            for (int r = -2; r < reps; ++r) {
                if (r == 0) {
                    MPI_Barrier(MPI_COMM_WORLD);
                    start = MPI_Wtime();
                }
                if (rank == 0) {
                    send_msg(p, &b, bytes, peer);
                    recv_msg(p, &b, bytes, peer);
                } else {
                    recv_msg(p, &b, bytes, peer);
                    send_msg(p, &b, bytes, peer);
                }
            }
            const double t = (MPI_Wtime() - start) / (2.0 * reps);      // one way: half a round trip
            if (rank == 0)
                printf(" %10.2f", 1e6 * t);
        }
        if (rank == 0) {
            printf("\n");
            fflush(stdout);
        }
    }

    CUDA_CHECK(cudaFree(b.dev));
    CUDA_CHECK(cudaFreeHost(b.pinned));
    CUDA_CHECK(cudaFree(b.managed));
    free(b.host);
    MPI_Finalize();
    return 0;
}

In [ ]:
# time per message and bandwidth of the datapaths (requires matplotlib)
import matplotlib.pyplot as plt

surface, ink, muted, grid, axis = '#fcfcfb', '#0b0b0b', '#52514e', '#e4e3df', '#a3a29c'
colors = {'host': '#52514e', 'pageable': '#eb6834', 'pinned': '#eda100', 'managed': '#8a5cc2', 'device': '#2a78d6'}

lines = pingpong.splitlines() if 'pingpong' in globals() else []
header = next((l.split() for l in lines if l.lstrip().startswith('bytes')), [])
paths = [p for p in header[1:] if p in colors]
rows = [l.split() for l in lines if l.split() and l.split()[0].isdigit()]

if not rows:
    print('No measurement - run the ping-pong cell first (it needs a GPU)')
else:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.4), facecolor=surface)
    for p, name in enumerate(paths):
        pts = [(int(r[0]), float(r[1 + p])) for r in rows if len(r) > 1 + p and r[1 + p] != 'n/a']
        if pts:
            axes[0].plot(*zip(*pts), marker='o', ms=4, color=colors[name], label=name)
            axes[1].plot([s for s, _ in pts], [s / t / 1e3 for s, t in pts], marker='o', ms=4, color=colors[name], label=name)
    for ax, title in zip(axes, ('Time per message [µs]', 'Bandwidth [GB/s]')):
        ax.set_facecolor(surface)
        ax.set_xscale('log', base=2)
        ax.set_yscale('log')
        ax.set_xlabel('message size [bytes]', color=muted)
        ax.set_title(title, loc='left', fontsize=12, color=ink)
        ax.grid(True, color=grid, linewidth=0.8)
        ax.set_axisbelow(True)
        ax.tick_params(colors=muted)
        for side in ('top', 'right'):
            ax.spines[side].set_visible(False)
        for side in ('left', 'bottom'):
            ax.spines[side].set_color(axis)
    axes[0].legend(frameon=False, labelcolor=ink)
    fig.tight_layout()
    plt.show()

What to look for — your numbers will differ:

* **Small messages:** the host baseline takes well under a microsecond within a node, 1–2 µs between nodes.
  Staging adds two `cudaMemcpy()` calls of several microseconds each: the latency grows several-fold.
  CUDA-aware MPI does better for small messages if it can use GDRCopy or CUDA IPC; otherwise it may be no faster than staging.
* **Large messages:** staging through pageable memory is slowest, pinned staging is limited by the sum of three transfers.
  Between two GPUs of a node with NVLink, CUDA-aware MPI reaches 100 GB/s and more — several times any path through the host.
  It rarely reaches the peak of NVLink: the library splits the message, and every piece costs synchronization.
  Between nodes, GPUDirect RDMA and pipelined staging get close to the bandwidth of the network.
* **Managed memory** pays for page migrations in both directions on every message — with a library that is not CUDA-aware, fault by fault on the CPU.
  A CUDA-aware library moves it with the copy engines and may even beat pinned staging for large messages.
* The **first messages** on a path are much slower: the library sets up connections, registers buffers with the NIC, opens CUDA IPC handles — and caches all of it.
  This is why the benchmark warms up, and why a real code should **allocate its communication buffers once and reuse them** instead of calling `cudaMalloc()` and `cudaFree()` for every message.
* With both processes on one GPU, the device path is a copy within one GPU memory — fast, but not representative.
* Steps or bumps in a curve, even in the host baseline, are usually the library switching protocols (eager, rendezvous) or transports at fixed message sizes.

## Synchronization: MPI Knows Nothing about Streams

Kernels and asynchronous copies run in CUDA **streams**, asynchronously to the host.
MPI calls are executed by the host and know nothing about streams: when `MPI_Send()` gets a device buffer, the library starts reading it — whether the kernel that writes it has finished or not.

The rules:

* **Before MPI reads** a device buffer, i.e. before a send, the work producing it must be complete: `cudaStreamSynchronize()` for the stream of the producing kernel, `cudaEventSynchronize()` for an event recorded after it, or `cudaDeviceSynchronize()`.
* **After a receive has completed**, the data is in GPU memory, and a kernel launched afterwards sees it.
* Staging is no exception: `cudaMemcpy()` waits only for the work in the default stream — not for streams created with `cudaStreamNonBlocking`, or with nvcc's option `--default-stream per-thread`.
* As for every nonblocking MPI operation: don't modify a send buffer or read a receive buffer on the GPU before the request has completed.

In the following cell, a slow kernel in a non-blocking stream computes the data that rank 0 then sends — first without, then with synchronization.

> 🤔 **Predict:** What does rank 1 receive in the first round?

In [ ]:
%%mpicuda -r -n 2 -i mpicuda.h -f Icode -o code/mpicuda-sync.cu
// the GPU's clock in nanoseconds: a special register, read with inline PTX assembly
__device__ unsigned long long globaltimer(void) {
    unsigned long long t;
    asm volatile("mov.u64 %0, %%globaltimer;" : "=l"(t));
    return t;
}

// a slow producer: computes for 100 ms, then writes its result
__global__ void produce(int *x, int n, int value) {
    const unsigned long long start = globaltimer();
    while (globaltimer() - start < 100000000ull)
        ;
    const int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n)
        x[i] = value;
}

int main(int argc, char *argv[]) {
    MPI_Init(&argc, &argv);
    int rank;
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    select_gpu();
    const char *how;
    const bool aware = mpi_cuda_aware(&how) == 1;   // otherwise: stage through host memory

    const int n = 1024;
    int *d_x, *h_x = (int *)malloc(n * sizeof(int));
    CUDA_CHECK(cudaMalloc(&d_x, n * sizeof(int)));
    CUDA_CHECK(cudaMemset(d_x, 0, n * sizeof(int)));
    CUDA_CHECK(cudaDeviceSynchronize());

    // a stream of the application, e.g. to overlap kernels; non-blocking: it does not
    // synchronize with the default stream, which cudaMemcpy() uses
    cudaStream_t stream;
    CUDA_CHECK(cudaStreamCreateWithFlags(&stream, cudaStreamNonBlocking));

    // warm up: on the first use of a buffer, the library sets up its path (registration, CUDA IPC
    // handles) - which may synchronize the whole GPU and would hide the bug in the first round
    if (rank == 0)
        MPI_Send(aware ? (void *)d_x : (void *)h_x, n, MPI_INT, 1, 0, MPI_COMM_WORLD);
    else if (rank == 1)
        MPI_Recv(aware ? (void *)d_x : (void *)h_x, n, MPI_INT, 0, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);

    if (rank == 0)
        printf("%s\n", aware ? "CUDA-aware MPI: sending the device buffer" : "not CUDA-aware: staging through host memory");
    for (int fixed = 0; fixed <= 1; ++fixed) {
        const int value = 42 + fixed;
        if (rank == 0) {
            produce<<<n / 256, 256, 0, stream>>>(d_x, n, value);
            CUDA_CHECK_LAUNCH();
            if (fixed)
                CUDA_CHECK(cudaStreamSynchronize(stream));          // wait until the data is there
            if (aware)
                MPI_Send(d_x, n, MPI_INT, 1, 0, MPI_COMM_WORLD);
            else {
                CUDA_CHECK(cudaMemcpy(h_x, d_x, n * sizeof(int), cudaMemcpyDeviceToHost));
                MPI_Send(h_x, n, MPI_INT, 1, 0, MPI_COMM_WORLD);
            }
            CUDA_CHECK(cudaStreamSynchronize(stream));              // before the buffer is reused
        } else if (rank == 1) {
            if (aware)
                MPI_Recv(d_x, n, MPI_INT, 0, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
            else {
                MPI_Recv(h_x, n, MPI_INT, 0, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
                CUDA_CHECK(cudaMemcpy(d_x, h_x, n * sizeof(int), cudaMemcpyHostToDevice));
            }
            // check on the host what arrived in GPU memory
            CUDA_CHECK(cudaMemcpy(h_x, d_x, n * sizeof(int), cudaMemcpyDeviceToHost));
            int correct = 0;
            for (int i = 0; i < n; ++i)
                correct += h_x[i] == value;
            printf("%-28s: %4d of %d values correct (x[0] = %d, expected %d)\n",
                   fixed ? "with cudaStreamSynchronize()" : "without synchronization", correct, n, h_x[0], value);
        }
    }

    CUDA_CHECK(cudaStreamDestroy(stream));
    CUDA_CHECK(cudaFree(d_x));
    free(h_x);
    MPI_Finalize();
    return 0;
}

Without synchronization, rank 1 receives the zeros that were in the buffer before the kernel ran — no error, no warning, just wrong data.
With a faster kernel, the result changes from run to run: the most common bug in MPI+CUDA codes.

The program sends one message before the demonstration.
Without it, the CUDA-aware version can deliver the right values: on the first use of a device buffer, the library sets up the path for it (memory registration, CUDA IPC handles), and some of these CUDA calls synchronize the whole GPU — the kernel finishes by accident.
The bug then only shows up from the second message on, a good reason to test with more than one iteration.

The synchronization serializes the GPU and MPI: the host waits for the GPU, then the GPU idles while MPI communicates.
Hiding the communication needs more structure — see the halo exchange below —, or communication that is itself ordered into the streams:

* **NCCL**, the NVIDIA Collective Communications Library, offers collectives and point-to-point operations that are enqueued in a stream like a kernel; it is the standard for deep learning.
  A typical program starts with MPI and uses NCCL for the GPU data (see the collectives below).
* **NVSHMEM** provides a partitioned global address space across GPUs: kernels read and write the memory of other GPUs themselves, without the CPU.
* **Stream-aware MPI:** MPICH (4.1 and later) has extensions that enqueue MPI operations into a CUDA stream, `MPIX_Stream_comm_create()` together with `MPIX_Send_enqueue()` and `MPIX_Recv_enqueue()`; the MPI Forum discusses standardizing such an interface.

## Halo Exchange and Collectives with GPU Data

A stencil code exchanges halos in every iteration, like the ghost cells in [MPI Point-to-Point Communication](point-to-point.ipynb).
On GPUs, three things change.

**Non-contiguous halos.** With a 2D decomposition, the halo columns are strided.
CUDA-aware libraries accept derived datatypes for device buffers, but implementations differ widely: some pack on the GPU, others issue one small copy per block — painfully slow for a column of single values.
The robust solution is a **pack kernel** that gathers the column into a contiguous device buffer, and an unpack kernel on the receiver:

```cpp
// gathers column i of u (leading dimension ld) into buf
__global__ void pack_column(double *buf, const double *u, int i, int jmax, int ld) {
    const int j = blockIdx.x * blockDim.x + threadIdx.x;
    if (j < jmax)
        buf[j] = u[(size_t)(j + 1) * ld + i];
}
```

**Synchronization.** Before the halos are sent, the sweep that computed them must have finished — see above.

**Overlap.** Only the cells next to the halos are needed by the neighbors.
Compute them first, and communicate while the GPU computes the interior:

```cpp
boundary<<<…, 0, s_boundary>>>(…);      // the cells next to the halos
interior<<<…, 0, s_interior>>>(…);      // the rest: runs on while MPI communicates
cudaStreamSynchronize(s_boundary);      // the data for the neighbors is ready
// pack kernels, MPI_Irecv() and MPI_Isend() of the halos, MPI_Waitall(), unpack kernels
cudaStreamSynchronize(s_interior);      // the sweep is complete
```

With staging, the copies go into the boundary stream as `cudaMemcpyAsync()` with pinned buffers.
Exercise 3 starts with the simple version.

**Reductions.** Iterative solvers need global sums and maxima: residual norms, dot products.
For a scalar, the simplest and usually fastest way is to reduce on the GPU to one value per process, copy these 8 bytes to the host, and reduce there:

```cpp
reduce<<<…>>>(d_result, …);             // one value per process, see the reduction in the CUDA primer
cudaMemcpy(&local, d_result, sizeof(double), cudaMemcpyDeviceToHost);
MPI_Allreduce(&local, &global, 1, MPI_DOUBLE, MPI_SUM, comm);
```

CUDA-aware libraries accept device buffers in collectives, too, but some apply the reduction operation on the CPU and copy the data back and forth — irrelevant for a scalar, expensive for large vectors.

**Large collectives on GPU data** — like the allreduce of the gradients in deep learning — are the domain of **NCCL**: its collectives are optimized for NVLink, NVSwitch and GPUDirect RDMA, and they are enqueued in a stream.
MPI typically sets it up:

```cpp
ncclUniqueId id;
if (rank == 0)
    ncclGetUniqueId(&id);
MPI_Bcast(&id, sizeof(id), MPI_BYTE, 0, MPI_COMM_WORLD);   // every process needs the same id
ncclComm_t nccl;
ncclCommInitRank(&nccl, size, id, rank);
ncclAllReduce(d_send, d_recv, n, ncclDouble, ncclSum, nccl, stream);   // asynchronous, in the stream
```

## Which Datapath When?

| Path | Requirements | Small messages | Large messages | Remarks |
|---|---|---|---|---|
| staged, pageable buffer | any MPI | slow: + 2 × `cudaMemcpy()` | slowest: driver copies on top | only as a first step |
| staged, pinned buffer | any MPI | slow | sum of three transfers | the portable baseline; you control every step |
| pipelined staging, by hand | any MPI | slow | close to the slowest transfer | complex; a CUDA-aware library does it better |
| CUDA-aware: GPUDirect P2P, CUDA IPC | CUDA-aware MPI; GPUs with NVLink or a common PCIe root; IPC between the processes | fast | NVLink: hundreds of GB/s | the main argument for CUDA-aware MPI within a node |
| CUDA-aware: GPUDirect RDMA | CUDA-aware MPI and UCX or libfabric; `nvidia_peermem` or DMA-BUF; NIC close to the GPU | fast with GDRCopy | close to the network bandwidth | the main argument between nodes |
| unified memory | any MPI | slow: page faults | migrations in both directions | convenient for porting, not for communication buffers |
| NCCL, NVSHMEM | NVIDIA libraries | fast | best for collectives | beyond MPI: ordered in streams or started by kernels |

> **Conclusion:** keep the data on the GPU, and let a CUDA-aware MPI library move it — where it is available, it is both simpler and faster.
> Write staged communication as a portable fallback, with pinned buffers and correct synchronization.
> And measure what you get: a CUDA-aware library that silently falls back to staging, e.g. because of the GPU binding or a missing kernel module, is a common surprise.

## Exercises

### Exercise 1: Datapaths on Paper — and on Your Node

A process on GPU 0 of node A (see the figure of the node) sends a halo of 1 MiB to a process on another GPU.
Assume 50 GB/s for PCIe and the network, 900 GB/s for NVLink, 20 GB/s for an MPI message between two processes of a node through host memory, and latencies of 2 µs per MPI message and 5 µs per `cudaMemcpy()`.

1. Estimate the time of the message to GPU 1 of the same node (a) staged through pinned host memory, (b) with CUDA-aware MPI over NVLink.
2. The same for the GPU of node B: (a) staged, (b) GPUDirect RDMA through NIC 0, (c) pipelined staging in 8 chunks, neglecting the latencies of the chunks.
3. Why should the process on GPU 2 not send through NIC 0?
4. Run `nvidia-smi topo -m` on your GPU node: which NIC should each GPU use? Run the ping-pong with the two processes on different GPUs, and compare the result for 1 MiB with your estimates.

<details>
<summary>Solution</summary>

1 MiB ≈ 1.05 MB: 21 µs over PCIe or the network, 1.2 µs over NVLink, 52 µs through host memory within a node.

1. (a) Staged: (5 + 21) + (2 + 52) + (5 + 21) ≈ **106 µs**. (b) CUDA-aware over NVLink: 2 + 1.2 ≈ **3 µs** — about 35 times faster.
2. (a) Staged: 26 + (2 + 21) + 26 ≈ **75 µs**.
   (b) GPUDirect RDMA: the data flows from the GPU through the PCIe switch to the NIC, over the network and on to the other GPU, in one pipelined transfer: 2 + 21 ≈ **23 µs**.
   (c) Pipelined: all three stages run at 50 GB/s and overlap; the time is that of one stage plus the two other stages of one chunk: 21 + 2 × 2.6 ≈ **26 µs**, plus the latencies of the first chunk's three steps (5 + 2 + 5 µs) ≈ **38 µs** — half of (a).
   In reality, the latencies of the chunks limit how small chunks can be — libraries use chunks of a few hundred KiB.
3. The path from GPU 2 to NIC 0 crosses the inter-socket link (`SYS` in `nvidia-smi topo -m`).
   GPUDirect RDMA across it is slow, and many libraries don't even try it, but stage through host memory; the traffic also competes with everything else that crosses the sockets.
   NIC 1, on the same PCIe switch (`PIX`), is the right one.
4. Depends on your system. With UCX, the variable `UCX_NET_DEVICES` selects the NIC per process, e.g. set by a wrapper script; Slurm and some MPI libraries choose the closest NIC automatically.
</details>

### Exercise 2: Pipelined Staging *(advanced)*

The cell sends 64 MiB from GPU memory of rank 0 to GPU memory of rank 1, staged through pinned host memory — first blocking, then pipelined in 1, 4, 16 and 64 chunks.
Implement `pipelined()`:

* Rank 0 enqueues the copies of all chunks to the host in one stream, with an event after each, and sends chunk `k` with `MPI_Isend()` as soon as its event has completed.
* Rank 1 starts an `MPI_Irecv()` for every chunk, and enqueues the copy of chunk `k` to the GPU as soon as it has arrived.

Start by waiting for the events with `cudaEventSynchronize()`.
If the pipeline is no faster than the blocking version, think about who moves the data of a large message, and when — and wait with `cudaEventQuery()` instead, calling `MPI_Testall()` on the chunks already sent in between.

How many chunks are best, and why not more?
Compare the bandwidth with the columns `pinned` and `device` of the ping-pong.

<details>
<summary>Solution</summary>

```cpp
static void pipelined(char *d_buf, char *h_buf, size_t n, int nchunks, cudaStream_t stream, cudaEvent_t *done, int rank) {
    const size_t chunk = n / nchunks;               // n is a multiple of nchunks
    MPI_Request *requests = (MPI_Request *)malloc(nchunks * sizeof(MPI_Request));
    if (rank == 0) {
        for (int k = 0; k < nchunks; ++k) {         // enqueue all copies, one event after each
            CUDA_CHECK(cudaMemcpyAsync(h_buf + k * chunk, d_buf + k * chunk, chunk, cudaMemcpyDeviceToHost, stream));
            CUDA_CHECK(cudaEventRecord(done[k], stream));
        }
        for (int k = 0; k < nchunks; ++k) {         // send each chunk as soon as it is on the host
            int flag;
            while (cudaEventQuery(done[k]) == cudaErrorNotReady)
                MPI_Testall(k, requests, &flag, MPI_STATUSES_IGNORE);   // progress the chunks already sent
            MPI_Isend(h_buf + k * chunk, (int)chunk, MPI_BYTE, 1, k, MPI_COMM_WORLD, &requests[k]);
        }
        MPI_Waitall(nchunks, requests, MPI_STATUSES_IGNORE);
    } else {
        for (int k = 0; k < nchunks; ++k)
            MPI_Irecv(h_buf + k * chunk, (int)chunk, MPI_BYTE, 0, k, MPI_COMM_WORLD, &requests[k]);
        for (int k = 0; k < nchunks; ++k) {         // copy each chunk to the GPU as soon as it has arrived
            MPI_Wait(&requests[k], MPI_STATUS_IGNORE);
            CUDA_CHECK(cudaMemcpyAsync(d_buf + k * chunk, h_buf + k * chunk, chunk, cudaMemcpyHostToDevice, stream));
        }
        CUDA_CHECK(cudaStreamSynchronize(stream));
    }
    free(requests);
}
```

**Progress.** With `cudaEventSynchronize()` instead of the polling loop, the pipeline may gain nothing at all — on an H100 node with Open MPI and UCX, all chunk counts take as long as the blocking version.
Messages of several MiB use the rendezvous protocol, and depending on the transport, the library moves the data only while the *sender* is inside an MPI call.
`cudaEventSynchronize()` waits in CUDA: the sends of the first chunks don't start before rank 0 calls `MPI_Waitall()` — after all copies are done.
Polling the event and calling `MPI_Testall()` in between keeps the transfers going; on that node, 3.9 ms become 2.2–3.1 ms with 16 to 64 chunks, depending on where the processes run relative to their GPUs.

With one chunk, the pipeline is the blocking version.
With more chunks, the three stages overlap, and the time approaches that of the slowest stage — up to three times faster if all stages are equally fast, as between nodes, where PCIe and the network have similar bandwidths.
Within a node, the MPI stage — a copy through host memory — competes with the two PCIe copies for the memory bandwidth of the host, and the pipeline gains less.
Too many chunks don't pay: every chunk adds the overhead of a copy, an event and an MPI message.

CUDA-aware libraries do exactly this internally, with tuned chunk sizes, their own progress and without the overhead of separate MPI messages — one more reason to use them.
</details>

In [ ]:
%%mpicuda -r -n 2 -i mpicuda.h -f Icode -o code/mpicuda-ex-pipeline.cu
#include <string.h>

// Sends n bytes from GPU memory of rank 0 to GPU memory of rank 1, staged through pinned host memory.

// blocking: copy everything, send everything, copy everything
static void staged(char *d_buf, char *h_buf, size_t n, int rank) {
    if (rank == 0) {
        CUDA_CHECK(cudaMemcpy(h_buf, d_buf, n, cudaMemcpyDeviceToHost));
        MPI_Send(h_buf, (int)n, MPI_BYTE, 1, 0, MPI_COMM_WORLD);
    } else {
        MPI_Recv(h_buf, (int)n, MPI_BYTE, 0, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
        CUDA_CHECK(cudaMemcpy(d_buf, h_buf, n, cudaMemcpyHostToDevice));
    }
}

// pipelined: in nchunks chunks, so that the copies and the transfers of different chunks overlap
static void pipelined(char *d_buf, char *h_buf, size_t n, int nchunks, cudaStream_t stream, cudaEvent_t *done, int rank) {
    const size_t chunk = n / nchunks;               // n is a multiple of nchunks
    MPI_Request *requests = (MPI_Request *)malloc(nchunks * sizeof(MPI_Request));
    (void)chunk;                                    // silences the compiler warning until you use it
    if (rank == 0) {
        // TODO: enqueue the copies of all chunks to the host in the stream, with an event after each;
        //       wait for the event of chunk k (cudaEventSynchronize - or better: poll it with
        //       cudaEventQuery and call MPI_Testall on the chunks sent so far), then start its
        //       MPI_Isend(); finally, wait for all sends
    } else {
        // TODO: start an MPI_Irecv() for every chunk; wait for the chunks in order
        //       (MPI_Wait) and enqueue the copy of each one to the GPU as soon as it has arrived;
        //       finally, wait for the stream
    }
    free(requests);
}

int main(int argc, char *argv[]) {
    MPI_Init(&argc, &argv);
    int rank;
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    select_gpu();

    const size_t n = (size_t)64 << 20;              // 64 MiB
    const int reps = 10, max_chunks = 64;
    char *d_buf, *h_buf;
    CUDA_CHECK(cudaMalloc(&d_buf, n));
    CUDA_CHECK(cudaMallocHost(&h_buf, n));
    cudaStream_t stream;
    CUDA_CHECK(cudaStreamCreate(&stream));
    cudaEvent_t done[max_chunks];                   // done[k]: chunk k has been copied
    for (int k = 0; k < max_chunks; ++k)
        CUDA_CHECK(cudaEventCreateWithFlags(&done[k], cudaEventDisableTiming));

    for (int nchunks = 0; nchunks <= max_chunks; nchunks = nchunks ? 4 * nchunks : 1) {
        // the data to send: byte i is (unsigned char)(i * 7)
        unsigned char *data = (unsigned char *)h_buf;
        for (size_t i = 0; i < n; ++i)
            data[i] = (unsigned char)(rank == 0 ? i * 7 : 0);
        CUDA_CHECK(cudaMemcpy(d_buf, h_buf, n, cudaMemcpyHostToDevice));
        memset(h_buf, 0, n);

        double t = 0.0;
        for (int r = -1; r < reps; ++r) {           // one round to warm up
            MPI_Barrier(MPI_COMM_WORLD);
            const double start = MPI_Wtime();
            if (nchunks == 0)
                staged(d_buf, h_buf, n, rank);
            else
                pipelined(d_buf, h_buf, n, nchunks, stream, done, rank);
            MPI_Barrier(MPI_COMM_WORLD);
            if (r >= 0)
                t += MPI_Wtime() - start;
        }
        t /= reps;

        // check what arrived on the GPU of rank 1
        size_t wrong = 0;
        if (rank == 1) {
            CUDA_CHECK(cudaMemcpy(h_buf, d_buf, n, cudaMemcpyDeviceToHost));
            for (size_t i = 0; i < n; ++i)
                wrong += data[i] != (unsigned char)(i * 7);
        }
        MPI_Bcast(&wrong, sizeof(wrong), MPI_BYTE, 1, MPI_COMM_WORLD);
        if (rank == 0) {
            if (nchunks == 0)
                printf("%-22s", "blocking");
            else
                printf("pipelined, %2d chunks  ", nchunks);
            printf(": %7.2f ms, %5.1f GB/s, %zu wrong bytes\n", 1e3 * t, n / t / 1e9, wrong);
        }
    }

    for (int k = 0; k < max_chunks; ++k)
        CUDA_CHECK(cudaEventDestroy(done[k]));
    CUDA_CHECK(cudaStreamDestroy(stream));
    CUDA_CHECK(cudaFree(d_buf));
    CUDA_CHECK(cudaFreeHost(h_buf));
    MPI_Finalize();
    return 0;
}

### Exercise 3: Jacobi on Several GPUs

The cell distributes the Jacobi solver of the [CUDA primer](cuda-primer.ipynb) (Exercise 3) over the processes with a 1D decomposition in `j`: every process owns a block of rows, and the halos are contiguous rows.
It runs 2000 sweeps on 2048 × 2048 cells; the boundary values are $u = x + y$, which is also the exact solution.
Everything is in place except `halo_exchange()`.
It is called once before the time is taken: the first messages set up the library's paths, which can take several 100 ms — more than all sweeps together.

1. Implement it for both cases: directly on the device buffers with a CUDA-aware library, otherwise staged through the pinned host buffers.
   The mean error must be the same with `-n 1`, `-n 2` and `-n 4` — after 2000 sweeps, it is still far from zero; what counts is that it does not depend on the number of processes.
2. Compare the runtimes for one and several processes, with and without CUDA-awareness (`-e PAMPI_CUDA_AWARE=0` forces staging). How long does one sweep take, and how long the halo exchange?
   Use at most as many processes as GPUs for timings: processes sharing a GPU take turns.
3. *(advanced)* Overlap the halo exchange with the computation: compute the two outer rows in one stream and the interior in another, and exchange the new halos while the interior is computed.

<details>
<summary>Solution</summary>

```cpp
static void halo_exchange(double *d_u, double *h_send, double *h_recv, int imax, int jmaxLocal,
                          int below, int above, bool aware) {
    const size_t row = imax * sizeof(double);
    if (aware) {
        CUDA_CHECK(cudaDeviceSynchronize());        // MPI knows nothing about the sweep still running
        MPI_Sendrecv(d_u + IDX(1, jmaxLocal), imax, MPI_DOUBLE, above, 0,
                     d_u + IDX(1, 0), imax, MPI_DOUBLE, below, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
        MPI_Sendrecv(d_u + IDX(1, 1), imax, MPI_DOUBLE, below, 1,
                     d_u + IDX(1, jmaxLocal + 1), imax, MPI_DOUBLE, above, 1, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
    } else {
        // cudaMemcpy() in the default stream waits for the sweep; no copies for MPI_PROC_NULL
        if (above != MPI_PROC_NULL)
            CUDA_CHECK(cudaMemcpy(h_send, d_u + IDX(1, jmaxLocal), row, cudaMemcpyDeviceToHost));
        MPI_Sendrecv(h_send, imax, MPI_DOUBLE, above, 0, h_recv, imax, MPI_DOUBLE, below, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
        if (below != MPI_PROC_NULL)                 // nothing received: keep the boundary values
            CUDA_CHECK(cudaMemcpy(d_u + IDX(1, 0), h_recv, row, cudaMemcpyHostToDevice));
        if (below != MPI_PROC_NULL)
            CUDA_CHECK(cudaMemcpy(h_send, d_u + IDX(1, 1), row, cudaMemcpyDeviceToHost));
        MPI_Sendrecv(h_send, imax, MPI_DOUBLE, below, 1, h_recv, imax, MPI_DOUBLE, above, 1, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
        if (above != MPI_PROC_NULL)
            CUDA_CHECK(cudaMemcpy(d_u + IDX(1, jmaxLocal + 1), h_recv, row, cudaMemcpyHostToDevice));
    }
}
```

* With CUDA-aware MPI, the `cudaDeviceSynchronize()` is essential: the sweep runs asynchronously, and without it MPI may send the old row — wrong results, but only sometimes.
  The staged version doesn't need it, since `cudaMemcpy()` in the default stream waits for the sweep.
* At the boundary of the domain, the neighbor is `MPI_PROC_NULL` and nothing is received: copying `h_recv` into the halo would overwrite the boundary values.
  Nothing is sent either: the copies to the host can be skipped — with one process, the staged version otherwise pays for two useless copies per sweep.
* A sweep moves 16 bytes per cell, 67 MB in total — about 10–40 µs on current GPUs, depending on their memory bandwidth.
  That is the same order of magnitude as the halo exchange, and the computation per GPU shrinks with more GPUs while the exchange stays the same: strong scaling of small problems ends quickly on GPUs.
  GPUs need large subdomains: the faster the computation, the more the ratio of halo to inner cells matters.
* Measured on two H100 GPUs with NVLink: 76 ms with one process (85 ms with the `cudaDeviceSynchronize()` of the CUDA-aware branch); with two, 74 ms CUDA-aware, 117 ms staged.
  The sweep itself gets more than twice as fast with two GPUs: both arrays of a subdomain (2 × 17 MB) fit into the 50 MB L2 cache of an H100 — a superlinear effect that hides part of the communication.
* Without the exchange before the timed loop, the CUDA-aware version took 300–450 ms: the one-time setup of CUDA IPC, about 250 ms, was charged to the sweeps.
* Overlap: a kernel that updates a range of rows `j0 … j1`, launched for the rows `1` and `jmaxLocal` in stream `s_boundary` and for `2 … jmaxLocal − 1` in stream `s_interior`; after `cudaStreamSynchronize(s_boundary)`, exchange the halos of the **new** array with `MPI_Irecv()`/`MPI_Isend()`, then wait for MPI and for `s_interior` before swapping.
  The halos for the first sweep are exchanged once before the loop.
</details>

In [ ]:
%%mpicuda -r -n 2 -i math.h mpicuda.h -f Icode -o code/mpicuda-ex-jacobi.cu
// cell (i, j) of the local grid with imax x jmaxLocal inner cells and one layer of halo or
// boundary cells; i is the contiguous index, the grid is distributed in j: the halos are rows
#define IDX(i, j) ((size_t)(j) * (imax + 2) + (i))

// one Jacobi sweep, as in the CUDA primer
__global__ void jacobi(double *unew, const double *u, int imax, int jmax) {
    const int i = blockIdx.x * blockDim.x + threadIdx.x + 1;
    const int j = blockIdx.y * blockDim.y + threadIdx.y + 1;
    if (i <= imax && j <= jmax)
        unew[IDX(i, j)] = 0.25 * (u[IDX(i - 1, j)] + u[IDX(i + 1, j)] + u[IDX(i, j - 1)] + u[IDX(i, j + 1)]);
}

// exchanges the halo rows of d_u with the neighbors below and above (MPI_PROC_NULL at the boundary)
static void halo_exchange(double *d_u, double *h_send, double *h_recv, int imax, int jmaxLocal,
                          int below, int above, bool aware) {
    // TODO: send the top inner row (j = jmaxLocal) up and receive the bottom halo (j = 0) from below,
    //       then send the bottom inner row (j = 1) down and receive the top halo (j = jmaxLocal + 1)
    //       from above, each with one MPI_Sendrecv() of the imax values i = 1 ... imax:
    //       aware: directly from and to the rows in GPU memory
    //       else:  staged through the pinned host buffers h_send and h_recv (imax values each)
    //       Make sure that the GPU has finished the previous sweep before MPI reads d_u!
}

int main(int argc, char *argv[]) {
    MPI_Init(&argc, &argv);
    int rank, size;
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);
    select_gpu();
    const char *how;
    const bool aware = mpi_cuda_aware(&how) == 1;

    const int imax = 2048, jmax = 2048;             // global inner cells
    const int sweeps = 2000;                        // a fixed number, so that runtimes are comparable
    const double h = 1.0 / (imax + 1);

    // the rows of this process, and its neighbors
    const int jmaxLocal = jmax / size + (rank < jmax % size ? 1 : 0);
    const int joff = rank * (jmax / size) + (rank < jmax % size ? rank : jmax % size);
    const int below = rank > 0 ? rank - 1 : MPI_PROC_NULL, above = rank < size - 1 ? rank + 1 : MPI_PROC_NULL;
    const size_t bytes = (size_t)(imax + 2) * (jmaxLocal + 2) * sizeof(double);

    // initial values: u = x + y on the global boundary, 0 inside; the exact solution is x + y
    double *u = (double *)malloc(bytes);
    for (int j = 0; j <= jmaxLocal + 1; ++j)
        for (int i = 0; i <= imax + 1; ++i) {
            const int jg = joff + j;                // the global row
            u[IDX(i, j)] = (i == 0 || i == imax + 1 || jg == 0 || jg == jmax + 1) ? (i + jg) * h : 0.0;
        }

    double *d_u, *d_unew, *h_send, *h_recv;
    CUDA_CHECK(cudaMalloc(&d_u, bytes));
    CUDA_CHECK(cudaMalloc(&d_unew, bytes));
    CUDA_CHECK(cudaMemcpy(d_u, u, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_unew, u, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMallocHost(&h_send, imax * sizeof(double)));
    CUDA_CHECK(cudaMallocHost(&h_recv, imax * sizeof(double)));

    const dim3 threads(32, 8);
    const dim3 blocks((imax + threads.x - 1) / threads.x, (jmaxLocal + threads.y - 1) / threads.y);

    // the first messages set up the paths of the library (e.g. CUDA IPC), which can take several
    // 100 ms: exchange once before the time is taken - this also fills the halos for the first sweep
    halo_exchange(d_u, h_send, h_recv, imax, jmaxLocal, below, above, aware);
    MPI_Barrier(MPI_COMM_WORLD);
    const double start = MPI_Wtime();
    for (int s = 0; s < sweeps; ++s) {
        halo_exchange(d_u, h_send, h_recv, imax, jmaxLocal, below, above, aware);
        jacobi<<<blocks, threads>>>(d_unew, d_u, imax, jmaxLocal);
        CUDA_CHECK_LAUNCH();
        double *tmp = d_u; d_u = d_unew; d_unew = tmp;
    }
    CUDA_CHECK(cudaDeviceSynchronize());
    const double t = MPI_Wtime() - start;

    // the mean distance to the exact solution: the same for every number of processes
    CUDA_CHECK(cudaMemcpy(u, d_u, bytes, cudaMemcpyDeviceToHost));
    double local_err = 0.0, err;
    for (int j = 1; j <= jmaxLocal; ++j)
        for (int i = 1; i <= imax; ++i)
            local_err += fabs(u[IDX(i, j)] - (i + joff + j) * h);
    MPI_Reduce(&local_err, &err, 1, MPI_DOUBLE, MPI_SUM, 0, MPI_COMM_WORLD);
    if (rank == 0)
        printf("%d processes, %s: %d sweeps in %.1f ms, %.1f cell updates per ns, mean error %.10e\n", size,
               aware ? "CUDA-aware" : "staged", sweeps, 1e3 * t, (double)imax * jmax * sweeps / (t * 1e9), err / ((double)imax * jmax));

    CUDA_CHECK(cudaFree(d_u));
    CUDA_CHECK(cudaFree(d_unew));
    CUDA_CHECK(cudaFreeHost(h_send));
    CUDA_CHECK(cudaFreeHost(h_recv));
    free(u);
    MPI_Finalize();
    return 0;
}

## Summary

* Run **one MPI process per GPU**; select the GPU by the node rank before any other CUDA call, or let the launcher set `CUDA_VISIBLE_DEVICES`. Pin every process to the cores close to its GPU.
* A GPU node is a network of its own: NVLink between the GPUs is an order of magnitude faster than host memory, PCIe and the network.
  Data should take the shortest path: GPU to GPU over NVLink within a node, GPU to the NIC on its PCIe switch between nodes.
* **Staging** through host memory works with every MPI library, but adds two PCIe transfers and their latencies; use pinned buffers, and pipeline large messages.
* A **CUDA-aware** MPI library takes device pointers and chooses GPUDirect P2P and CUDA IPC within a node, GPUDirect RDMA or pipelined staging between nodes — if it is built and configured for it, and the kernel modules and the topology fit. Check, and measure.
* **Unified memory** works with any MPI library, but every message pays for page migrations.
* MPI knows nothing about CUDA streams: **synchronize** the producing stream before MPI reads a device buffer.
* Pack non-contiguous halos with kernels, overlap the exchange with the interior, reduce scalars on the GPU and then with MPI on the host, and use NCCL for large collectives on GPU data.

| Call / tool | Purpose |
|---|---|
| `MPI_Comm_split_type(…, MPI_COMM_TYPE_SHARED, …)` | number the processes of a node, to select their GPUs |
| `cudaGetDeviceCount()`, `cudaSetDevice()` | count the GPUs, select one |
| `cudaDeviceGetPCIBusId()`, `cudaDeviceCanAccessPeer()` | PCI address of a GPU; can two GPUs access each other's memory |
| `MPIX_Query_cuda_support()` (Open MPI), `MPIX_GPU_query_support()` (MPICH) | is the library CUDA-aware |
| `MPI_Comm_get_info()` with key `mpi_memory_alloc_kinds` | the memory kinds the library supports (MPI 4.1) |
| `cudaMallocHost()`, `cudaMallocManaged()` | pinned host memory for staging, unified memory |
| `cudaStreamSynchronize()`, `cudaEventRecord()`, `cudaEventSynchronize()`, `cudaEventQuery()` | wait for GPU work before MPI touches its data |
| `cudaStreamCreateWithFlags(…, cudaStreamNonBlocking)` | a stream that does not synchronize with the default stream |
| `ncclCommInitRank()`, `ncclAllReduce()` | NCCL: collectives on GPU data, ordered in a stream |
| `nvidia-smi topo -m` | the topology of a node |
| `CUDA_VISIBLE_DEVICES`, `srun --gpus-per-task=1 --gpu-bind=closest` | restrict and bind processes to GPUs |

[Back to the overview](../overview.ipynb)